# The critic as another model, and progress under a prover

`07`'s critic was a linear head on the policy's own hidden states (layer 23 of Qwen3.5-0.8B). Its PPO run from the recovery seed was the series' first to lift a greedy evaluation above step 0 (0.586 → 0.658 at step 40, 60 dev questions). This notebook keeps 07's seed, task, reward and PPO configuration and replaces the critic with **another model**, following

> Setlur, Nagpal, Fisch, Geng, Eisenstein, Agarwal, Agarwal, Berant, Kumar. *Rewarding Progress: Scaling Automated Process Verifiers for LLM Reasoning.* Google Research / Google DeepMind / CMU, arXiv [2410.08146](https://arxiv.org/abs/2410.08146) (ICLR 2025).

What the paper asks for, and what it means here:

| the paper | here |
|---|---|
| a step's process reward is its **progress** -- its advantage, the change in the chance of ending right -- not the value of the state it reaches | a turn's progress `A^μ = f(q after) − f(q before)`, `q` the critic's estimate of the base policy's chance of success |
| progress measured under a **prover** μ distinct from the base policy π; good provers are *complementary*: their advantages tell π's steps apart (distinguishability) and agree with π's own (alignment) -- its Theorem 3.1; too strong or too weak provers give `A^μ ≈ 0` | the prover is **BoK(π)**, the paper's class for its RL, with `Q^{BoK(π)} = 1 − (1 − Q^π)^K` (its Eq. 30); K chosen by the theorem's criterion on 07's episodes |
| the verifier that predicts progress (a PAV) is a **separate model** -- every PAV in the paper fine-tuned from Gemma 9B, for 2B, 9B and 27B policies | the critic is a separate pretrained model, frozen (fine-tuning a 2B model inside every PPO step is beyond this machine), read before each turn; two linear heads refitted at every step on the last 25 steps: v (PPO's baseline) and q (the success probability) |
| online RL with the effective reward `Q^π + α A^μ` per step | every turn's advantage gains `α A^μ` -- its own, not a sum to go |

The candidates: **Qwen3-1.7B** (another family: attention in every layer, a 151k vocabulary, against Qwen3.5's hybrid of gated delta-rule and attention layers with 248k; 2x the policy), **Qwen3.5-2B** (the policy's family one size up -- the paper's best prover across models was one size up), **Qwen3-4B** (the paper's verifier-to-policy ratio, 9B for 2B). And, for comparison, 07's critic's own features.

*(v1's results are in the Summary at the end: it tied 07 on test and holdout_shift and fell behind it on recovery from simulator errors. A second run with three of the 37 PPO implementation details is running.)*

In [1]:
import collections, importlib.util, json, math, sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "08_prover_critic.py").exists() else Path("experiments/notebooks/smoke_test35")
# This notebook's own module (07's, copied and changed as its header says); loaded by path because the file name starts
# with a digit. Nothing below needs WaterTAP or a GPU: every number is read from the runs' records.
spec = importlib.util.spec_from_file_location("pc08", HERE / "08_prover_critic.py")
pc = importlib.util.module_from_spec(spec)
sys.modules["pc08"] = pc
spec.loader.exec_module(pc)
RUNS, RUNS07 = pc.RUNS, pc.RUNS07
SEED07, CTRL07, PPO07 = "seed-n200-r100", "seed-n300-r0", "ppo-n200-r100-turn-s0"
MLXP = RUNS / "mlx-probes"  # 07's seed and PPO checkpoint through 08's probe, generated by MLX (Held out)


def rows_of(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def jload(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


FIT = jload(pc.CRITIC_FIT)
CHOICE = (FIT or {}).get("choice")
PPO08 = (f"ppo-{CHOICE['critic']}-bo{CHOICE['prover_k']}-a{CHOICE['alpha']}-s0" if CHOICE else None)
PPO08_D37 = PPO08.replace("-s0", "-d37-s0") if PPO08 else None  # the same with three of the 37 details (D37=1)
print("critic-fit:", "not run yet" if FIT is None else f"{len(FIT['candidates'])} candidates", "| choice:", CHOICE,
      "| 08's PPO runs:", PPO08, "and", PPO08_D37)

critic-fit: 4 candidates | choice: {'critic': 'qwen3-4b', 'layer': 27, 'ridge': 1.0, 'l2': 0.1, 'prover_k': 4, 'prover_k_criterion': 16, 'k_at_grid_edge': True, 'alpha': 0.8, 'tolerance': 0.005, 'online_q_logloss': {'qwen3-1.7b': 0.47628, 'qwen3.5-2b': 0.4577, 'qwen3-4b': 0.44818}} | 08's PPO runs: ppo-qwen3-4b-bo4-a0.8-s0 and ppo-qwen3-4b-bo4-a0.8-d37-s0


## critic-fit: which model reads the episodes best

The data are 07's own: every step of its PPO run from the recovery seed (8 episodes a step, sampled at T=1 on train questions) and, held out by question, the seed's and the control's 120 dev questions sampled once at T=1. Each episode is replayed through the simulators (`replay`: its calls run again, its budget and malformed-call errors given as the rollout gave them -- the replayed scores match the recorded rewards exactly) and read as **text** -- the prompt as the policy saw it, each turn, the tool responses -- since a separate model has its own tokenizer. A turn's state is the text before it; a candidate's features are its residual stream after a layer, at the last token before the turn.

Two heads per candidate layer: **v**, ridge regression on the turn's return to go (PPO's baseline, the job 07's head did), and **q**, L2 logistic regression on the episode's success -- the decision right, with the simulations that prove it run (the reward pays the decision in full only then): the base policy's `Q^π`, the prover's input. Penalties by 4-fold cross-validation, grouped by question. Two tests:

* **online** -- for every second step of 07's run, the heads fitted on the 25 steps before it and tested on its 8 episodes: the critic as PPO fits and uses it (a few hundred turns to fit on, a policy that keeps moving). The baselines: the window's success rate (`base`), and the turn index alone (`clock`: the window's mean by turn, 07's "position clock" for a turn);
* **held out** -- fitted on every train episode, tested on the dev questions.

The choice was fixed before the numbers: the separate candidate whose best layer has the lowest online q log-loss (the cheaper one within 0.005 nats).

In [2]:
def fmt(x, spec):
    return "-" if x is None else format(x, spec)


if FIT:
    d = FIT["data"]
    print(f"{d['train_episodes']} train episodes (07's steps {d['train_steps'][0]}-{d['train_steps'][1]}), {d['heldout_episodes']} held out, "
          f"{d['states']} turn states; success {d['success_train']:.3f} train, {d['success_heldout']:.3f} held out; replay: "
          f"{d['rewards_off_recorded']} rewards and {d['call_errors_off_recorded']} call errors off the recorded")
    print(f"\n{'candidate':12} {'layer':>5} | {'online: q logloss':>17} {'base':>6} {'clock':>6} {'AUC':>6} {'AUC t0':>6} {'AUC t>0':>7} "
          f"{'v EV':>6} {'clock':>6} | {'held out: q ll':>14} {'AUC':>6} {'v EV':>6} | {'s/episode':>9}")
    for name, c in FIT["candidates"].items():
        for L, r in c["layers"].items():
            on, ho = r["online"], r["heldout"]
            star = "*" if int(L) == c["best"]["layer"] else " "
            print(f"{name:12} {L:>4}{star} | {on['q_logloss']:17.4f} {on['q_logloss_base']:6.4f} {on['q_logloss_clock']:6.4f} "
                  f"{fmt(on['q_auc'], '6.3f')} {fmt(on['q_auc_first_turn'], '6.3f')} {fmt(on['q_auc_later_turns'], '7.3f')} "
                  f"{fmt(on['v_ev'], '+6.3f')} {fmt(on['v_ev_clock'], '+6.3f')} | {ho['q_logloss']:14.4f} {fmt(ho['q_auc'], '6.3f')} "
                  f"{fmt(ho['v_ev'], '+6.3f')} | {c['seconds_per_episode']:9.2f}")
    print("\n* each candidate's best layer (online q log-loss); 'policy' is 07's critic's features (07's seed, its own layers)")

848 train episodes (07's steps 0-105), 240 held out, 5165 turn states; success 0.366 train, 0.312 held out; replay: 0 rewards and 0 call errors off the recorded

candidate    layer | online: q logloss   base  clock    AUC AUC t0 AUC t>0   v EV  clock | held out: q ll    AUC   v EV | s/episode
policy         12  |            0.4641 0.6169 0.5818  0.823  0.639   0.857 +0.341 +0.210 |         0.3605  0.879 +0.443 |      2.78
policy         18  |            0.4414 0.6169 0.5818  0.843  0.710   0.869 +0.367 +0.210 |         0.3506  0.882 +0.453 |      2.78
policy         23* |            0.4408 0.6169 0.5818  0.844  0.721   0.869 +0.390 +0.210 |         0.3525  0.883 +0.464 |      2.78
qwen3-1.7b     14  |            0.4824 0.6169 0.5818  0.812  0.582   0.851 +0.327 +0.210 |         0.3724  0.866 +0.404 |      4.24
qwen3-1.7b     21* |            0.4763 0.6169 0.5818  0.816  0.607   0.855 +0.351 +0.210 |         0.3685  0.865 +0.422 |      4.24
qwen3-1.7b     28  |            0.4767 0.6169 

## The prover: which K

The paper's Theorem 3.1 bounds one policy update's improvement below by **distinguishability** -- the variance of `A^μ` over the base policy's actions at a state -- plus **alignment** -- the mean of `A^μ A^π`. For BoK(π) both are functions of `q` alone. With one recorded action per state, an action's deviation from its state's mean is taken to first order, `f_K'(q_t)(q_{t+1} − q_t)` (`q` is a martingale under π), and exactly `y − q` at the last turn; the plain variance over all turns would add the spread *between* states of `A^μ`'s mean and grow without end in K. Remark 3.1's reading: K > 1 helps a base policy that rarely succeeds (`Q = O(1/K)`), and a large K hurts where it already succeeds (`f_K` saturates). K is the maximiser of D + C on the chosen critic's online predictions; **α** scales `A^μ`'s spread to PPO's own advantage's (the paper tuned α on validation runs, finding 0.5-6 all helpful; a run here is a day).

In [3]:
if CHOICE:
    c = FIT["candidates"][CHOICE["critic"]]["layers"][str(CHOICE["layer"])]
    for test in ("online", "heldout"):
        tab = c["prover"][test]
        print(f"{CHOICE['critic']} layer {CHOICE['layer']}, {test}:")
        print(f"  {'K':>3} {'D':>8} {'C':>8} {'D+C':>8} {'mean A':>8} {'sd A':>7} {'won':>6} {'corr PPO adv':>12} {'alpha':>6}")
        for k, r in tab.items():
            print(f"  {k:>3} {r['D']:8.4f} {r['C']:8.4f} {r['D_plus_C']:8.4f} {r['mean']:+8.4f} {r['sd']:7.4f} {r['saturated']:6.3f} "
                  f"{fmt(r['corr_ppo_adv'], '12.3f')} {fmt(r['alpha_matching_sd'], '6.2f')}")
    print("\nchoice:", CHOICE)
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for name, cand in FIT["candidates"].items():
            Ls = sorted(cand["layers"], key=int)
            axes[0].plot([int(L) / cand["depth"] for L in Ls], [cand["layers"][L]["online"]["q_logloss"] for L in Ls], "o-", label=name)
        base = next(iter(next(iter(FIT["candidates"].values()))["layers"].values()))["online"]["q_logloss_base"]
        axes[0].axhline(base, color="gray", ls=":", label="window success rate")
        axes[0].set_xlabel("layer / depth"); axes[0].set_ylabel("online q log-loss"); axes[0].legend(fontsize=8)
        tab = c["prover"]["online"]
        ks = [int(k) for k in tab]
        for key, lab in (("D", "distinguishability D"), ("C", "alignment C"), ("D_plus_C", "D + C")):
            axes[1].plot(ks, [tab[k][key] for k in tab], "o-", label=lab)
        axes[1].set_xscale("log", base=2); axes[1].set_xlabel("K of the prover BoK(pi)"); axes[1].legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(HERE / "08_prover_critic_fit.png", dpi=90)
        plt.close(fig)
        print("figure: 08_prover_critic_fit.png")
    except ImportError:
        pass

qwen3-4b layer 27, online:
    K        D        C      D+C   mean A    sd A    won corr PPO adv  alpha
    1   0.0607   0.0607   0.1215  +0.0043  0.2464  0.005        0.527   1.03
    2   0.0824   0.0664   0.1489  -0.0363  0.2837  0.083        0.509   0.90
    3   0.1006   0.0659   0.1666  -0.0607  0.3005  0.173        0.486   0.84
    4   0.1155   0.0640   0.1795  -0.0768  0.3095  0.245        0.466   0.82
    6   0.1387   0.0596   0.1983  -0.0963  0.3191  0.331        0.438   0.80
    8   0.1578   0.0559   0.2137  -0.1073  0.3243  0.394        0.417   0.78
   16   0.2311   0.0470   0.2781  -0.1237  0.3334  0.538        0.371   0.76
qwen3-4b layer 27, heldout:
    K        D        C      D+C   mean A    sd A    won corr PPO adv  alpha
    1   0.0452   0.0452   0.0905  -0.0048  0.2126  0.000        0.574   1.07
    2   0.0581   0.0492   0.1074  -0.0458  0.2436  0.053        0.562   0.94
    3   0.0646   0.0494   0.1140  -0.0717  0.2618  0.109        0.532   0.87
    4   0.0680   0.04

figure: 08_prover_critic_fit.png


## PPO with the separate critic

07's configuration, copied: the turn as the unit of credit (λ 0.95 per turn), lr 2e-6, KL 0.01 to the seed, the turn ends and the answer's scaffolding out of the actor's loss, 3,072 policy tokens, greedy evaluations of 60 dev questions every 40 steps, and 07's early stops (the reward gate, the stall gate at actor steps 40 and 80, the no-regression gate, answer collapse). Changed: the critic -- the chosen model, frozen, its heads refitted every step on the last 25 steps' turns -- and every turn's advantage gains `α A^μ` under BoK(π).

The first 25 steps are **07's own** (`--warm-from`): the same seed draws the same questions, and in those steps the policy is not updated (the critic's warm-up), so their episodes, replayed, fill the new critic's window, and 07's step-0 evaluation is this run's. From step 25 on both runs see the same question at every step; the evaluations fall at the same steps.

From step 80 on, the run's rollouts and its evaluation at step 120 were generated by MLX (next section): it was stopped after its step-80 evaluation and continued from its saved state with `--generator mlx`, the update unchanged. Its evaluations at steps 40 and 80 are PyTorch's. A second run, D37 (below), takes three of the 37 PPO implementation details; it is listed here beside v1 and 07 as it comes in.

In [4]:
def rolling(xs, k=10):
    out = []
    for i in range(len(xs)):
        w = [x for x in xs[max(0, i - k + 1): i + 1] if x is not None]
        out.append(sum(w) / len(w) if w else None)
    return out


def median(xs):
    xs = sorted(xs)
    return (xs[(len(xs) - 1) // 2] + xs[len(xs) // 2]) / 2 if xs else float("nan")


def load_run(d):
    if not (d / "metrics.jsonl").exists():
        return None
    return {"m": rows_of(d / "metrics.jsonl"), "ev": rows_of(d / "eval.jsonl") if (d / "eval.jsonl").exists() else [],
            "end": jload(d / "ending.json"), "cfg": jload(d / "config.json")}


def generator_of(run, step):
    """08: who generated a step's rollouts (rows written before the field: PyTorch); an evaluation at step s is
    generated after step s - 1, by its generator."""
    row = next((r for r in run["m"] if r["step"] == step), None)
    return (row or {}).get("generator", "torch")


def summary_lines(label, run):
    m, ev, end = run["m"], run["ev"], run["end"]
    print(f"{label}: {len(m)} steps" + (f"; ended at step {end['stopped_at']} ({end['reason']}), best evaluation step {end['best_step']} "
                                        f"({end['best_reward']:.4f})" if end else " (running)"))
    act = [r for r in m if not r["critic_warmup"]]
    for name, rows in (("actor steps", act), ("last 25 steps", act[-25:])):
        if rows:
            mean = lambda k: sum(r[k] for r in rows if r.get(k) is not None) / max(1, sum(r.get(k) is not None for r in rows))  # noqa: E731
            print(f"   {name:14} reward {mean('reward_mean'):.3f} decision {mean('decision_correct'):.3f} evidence {mean('evidence'):.3f} "
                  f"| erred {mean('rec_sim_erred'):.3f} recovered {mean('rec_recovered_given_error'):.3f} | kl {mean('kl_mean'):.4f} "
                  f"| critic ev {mean('value_ev'):+.3f} (clock {mean('value_ev_position'):+.3f})")
    for g in ("torch", "mlx"):
        rows = [r for r in act if r.get("generator", "torch") == g]
        if rows:
            print(f"   generated by {g:5}: steps {rows[0]['step']}-{rows[-1]['step']}, median step {median([r['step_seconds'] for r in rows]):.0f}s, "
                  f"of which generation {median([r['gen_seconds'] for r in rows]):.0f}s")
    for r in ev:
        rc = r.get("recovery", {})
        g = rc.get("recovered_given_error")
        by = "  (07's)" if r.get("reused") else ("  (MLX)" if r["step"] and generator_of(run, r["step"] - 1) == "mlx" else "")
        print(f"     eval @{r['step']:>4}: reward {r['reward']:.4f} decision {r['decision_correct']:.3f} evidence {r['evidence']:.3f} "
              f"answers {r['answer_found']:.3f} | erred {rc.get('sim_erred', float('nan')):.3f} recovered {'-' if g is None else f'{g:.3f}'}{by}")


def paired_by_step(A, B):
    """B's sampled reward minus A's over the actor steps both ran: the same question at every step (the same rng)."""
    a = {r["step"]: r["reward_mean"] for r in A["m"] if not r["critic_warmup"]}
    b = {r["step"]: r["reward_mean"] for r in B["m"] if not r["critic_warmup"]}
    common = sorted(set(a) & set(b))
    if not common:
        return None
    diffs = [b[s] - a[s] for s in common]
    mu = sum(diffs) / len(diffs)
    sd = (sum((x - mu) ** 2 for x in diffs) / max(1, len(diffs) - 1)) ** 0.5
    return len(common), mu, mu - 1.96 * sd / len(diffs) ** 0.5, mu + 1.96 * sd / len(diffs) ** 0.5


R07 = load_run(RUNS07 / PPO07)
R08 = load_run(RUNS / PPO08) if PPO08 else None
R37 = load_run(RUNS / PPO08_D37) if PPO08_D37 else None
for label, run in ((f"07: {PPO07} (the policy's own critic)", R07), (f"08: {PPO08}", R08), (f"08, D37: {PPO08_D37}", R37)):
    if run:
        summary_lines(label, run)
    else:
        print(f"{label}: not run yet")
print()
for la, A, lb, B in (("07", R07, "08", R08), ("07", R07, "D37", R37), ("08", R08, "D37", R37)):
    p = paired_by_step(A, B) if A and B else None
    if p:
        print(f"sampled reward, {lb} - {la} over the {p[0]} actor steps both ran (same questions): {p[1]:+.4f} "
              f"(95% interval {p[2]:+.4f} to {p[3]:+.4f})")
for label, run in (("08", R08), ("D37", R37)):
    sep = [r for r in (run["m"] if run else []) if r.get("critic_ev_turn") is not None]
    if sep:
        mean = lambda k: sum(r[k] for r in sep if r.get(k) is not None) / len(sep)  # noqa: E731
        print(f"{label}'s critic, out of sample per step: ev (turn) {mean('critic_ev_turn'):+.3f}, q brier {mean('q_brier'):.3f} "
              f"(success {mean('success_rate'):.3f}), A^mu sd {mean('prog_std'):.3f} x alpha = {mean('prog_alpha_std'):.3f}, "
              f"PPO advantage sd {mean('adv_std'):.3f}")
if R07 and R08:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        runs = [(run, col, lab) for run, col, lab in ((R07, "tab:gray", "07"), (R08, "tab:blue", "08"), (R37, "tab:green", "08 D37")) if run]
        panels = [("reward_mean", "sampled reward"), ("decision_correct", "decision right"), ("evidence", "proving simulations"),
                  ("rec_recovered_given_error", "erred, then a clean simulate call"), ("kl_mean", "KL to the seed, per token"),
                  ("value_ev", "critic: explained variance (per token)")]
        fig, axes = plt.subplots(2, 3, figsize=(16, 7))
        for ax, (k, title) in zip(axes.flat, panels):
            for run, col, lab in runs:
                steps = [r["step"] for r in run["m"]]
                ys = [r.get(k) for r in run["m"]]
                ax.plot(steps, [y if y is not None else float("nan") for y in rolling(ys)], "-", color=col, label=lab)
            ax.axvline(25, color="k", lw=0.5, ls=":")
            ax.axvline(80, color="tab:blue", lw=0.5, ls=":")
            ax.set_title(title, fontsize=10)
        for run, col, lab in runs:
            for r in run["ev"]:
                axes.flat[0].plot(r["step"], r["reward"], "s", color=col)
        axes.flat[0].legend()
        fig.suptitle("Training samples, rolling 10 (07 gray, 08 blue, 08 D37 green); squares: greedy dev evaluations; "
                     "dotted: the actor's start (25) and 08's switch to MLX generation (80)")
        fig.tight_layout()
        fig.savefig(HERE / "08_prover_critic_curves.png", dpi=90)
        plt.close(fig)
        print("figure: 08_prover_critic_curves.png")
    except ImportError:
        pass

07: ppo-n200-r100-turn-s0 (the policy's own critic): 120 steps; ended at step 120 (step budget 120 reached), best evaluation step 120 (0.6730)
   actor steps    reward 0.538 decision 0.516 evidence 0.467 | erred 0.529 recovered 0.452 | kl 0.0019 | critic ev +0.246 (clock +0.084)
   last 25 steps  reward 0.552 decision 0.540 evidence 0.480 | erred 0.585 recovered 0.539 | kl 0.0030 | critic ev +0.208 (clock +0.091)
   generated by torch: steps 25-119, median step 474s, of which generation 148s
     eval @   0: reward 0.5856 decision 0.550 evidence 0.500 answers 0.633 | erred 0.583 recovered 0.400  (07's)
     eval @  40: reward 0.6580 decision 0.633 evidence 0.600 answers 0.717 | erred 0.483 recovered 0.448
     eval @  80: reward 0.6479 decision 0.633 evidence 0.583 answers 0.683 | erred 0.533 recovered 0.438
     eval @ 120: reward 0.6730 decision 0.683 evidence 0.633 answers 0.717 | erred 0.567 recovered 0.500
08: ppo-qwen3-4b-bo4-a0.8-s0: 120 steps; ended at step 120 (step budget 120

figure: 08_prover_critic_curves.png


## Generation on MLX

On this machine (Apple M3 Max, PyTorch on MPS) generation took about a third of a PPO step, and an evaluation of 60 questions about half an hour. From v1's step 80 on, generation runs in mlx-lm (`generator` in PPO_DEFAULTS, `--generator mlx`; `probe --generator mlx`). `MLXPolicy` loads the base model once and, before each rollout, folds the policy's current LoRA into it -- W = W0 + (α/r)·B·A in each of the 114 adapted linear layers, 0.27 s. `rollout_mlx` is `rollout` with the generation on MLX, each episode keeping its own prompt cache across its turns. The update, every log-probability and the critic stay PyTorch's: MLX only samples, and PPO's ratio is taken against PyTorch's log-probabilities of the sampled tokens (on a check, mean −0.051 nats per token, none outside the policy's support). Checked before the switch on four dev questions, greedy: 3 of 4 episodes token for token PyTorch's, the fourth the same for 4 of its 5 turns, with the same reward.

v1 was stopped after its step-80 evaluation and continued from its saved state with only the generator switched, so its steps before and after time the two generators on the same kind of work. The held-out probes below are all MLX's; the seed's probes, generated both ways, measure the generator's own difference.

In [5]:
if R08:
    act = [r for r in R08["m"] if not r["critic_warmup"]]
    for g in ("torch", "mlx"):
        rows = [r for r in act if r.get("generator", "torch") == g]
        if rows:
            print(f"v1, generated by {g:5}: {len(rows):2d} steps, median step {median([r['step_seconds'] for r in rows]):4.0f}s = generation "
                  f"{median([r['gen_seconds'] for r in rows]):4.0f}s + the rest {median([r['step_seconds'] - r['gen_seconds'] for r in rows]):4.0f}s "
                  f"| episode tokens {median([r['episode_tokens_mean'] for r in rows]):.0f}, policy tokens "
                  f"{median([r['policy_tokens_mean'] for r in rows]):.0f}")
print()
# The seed's probes generated both ways: 07's (PyTorch, the adapter merged) and here (MLX), on the same questions.
for tag in ("test-T0-n120-stop", "dev-T0-stop-forced", "dev-T1-n120-stop"):
    a, b = RUNS07 / SEED07 / "epoch1" / tag, MLXP / SEED07 / tag
    ja, jb = jload(f"{a}.json"), jload(f"{b}.json")
    if not (ja and jb):
        print(f"seed, {tag}: not run both ways yet")
        continue
    ea = {r["id"]: r for r in rows_of(f"{a}.episodes.jsonl")}
    eb = {r["id"]: r for r in rows_of(f"{b}.episodes.jsonl")}
    ids = sorted(set(ea) & set(eb))
    d = [eb[i]["score"]["total"] - ea[i]["score"]["total"] for i in ids]
    mu = sum(d) / len(d)
    se = (sum((x - mu) ** 2 for x in d) / len(d)) ** 0.5 / len(d) ** 0.5
    same = sum(ea[i]["turns"] == eb[i]["turns"] for i in ids)
    print(f"seed, {tag:19}: PyTorch {ja['overall']['reward']:.3f} in {ja['meta']['seconds']:5.0f}s | MLX {jb['overall']['reward']:.3f} in "
          f"{jb['meta']['seconds']:4.0f}s | MLX - PyTorch {mu:+.3f} (se {se:.3f}), the same reward on {sum(abs(x) < 1e-9 for x in d)}, "
          f"token for token the same episode on {same} of {len(ids)}" + (" (T=1: different draws)" if "-T1-" in tag else ""))

v1, generated by torch: 55 steps, median step  554s = generation  166s + the rest  382s | episode tokens 5112, policy tokens 918
v1, generated by mlx  : 40 steps, median step  452s = generation   53s + the rest  390s | episode tokens 5346, policy tokens 956

seed, test-T0-n120-stop  : PyTorch 0.543 in 10908s | MLX 0.564 in  832s | MLX - PyTorch +0.021 (se 0.015), the same reward on 100, token for token the same episode on 86 of 120
seed, dev-T0-stop-forced : PyTorch 0.686 in  3796s | MLX 0.688 in 1131s | MLX - PyTorch +0.002 (se 0.006), the same reward on 135, token for token the same episode on 118 of 144


seed, dev-T1-n120-stop   : PyTorch 0.430 in  3210s | MLX 0.496 in  985s | MLX - PyTorch +0.066 (se 0.030), the same reward on 21, token for token the same episode on 0 of 120 (T=1: different draws)


## Held out

The probes 07 ran for its own PPO run, on 08's checkpoints: greedy on the same 120 test questions and all 120 of holdout_shift, the forced probe (greedy, the 144 dev questions with a slip, each started from its slipped turns) and the 120 dev questions sampled once at T=1. Every probe here is generated by MLX -- 07's seed and 07's PPO checkpoint (step 120, its best) too, run through 08's probe: 08's prompts, tools, questions, forced prefill, budgets and scoring are 07's, function for function -- so all rows share the generator, and the differences are paired by question.

In [6]:
import random
splits = {s: pc.load_split(s) for s in ("dev", "test")}
TEST120 = {c["id"] for c in random.Random(0).sample(splits["test"], 120)}   # probe --limit 120's questions


def recov_line(o):
    r = o["recovery"]
    f = lambda x: "  -  " if x is None else f"{x:.3f}"  # noqa: E731
    return (f"n {o['n']:3d} reward {o['reward']:.3f} decision {o['decision_correct']:.3f} evidence {o['evidence']:.3f} "
            f"answers {o['answer_found']:.3f} | erred {r['sim_erred']:.3f} recovered {f(r['recovered_given_error'])} "
            f"repeats/ep {r['repeats_per_episode']:.2f}")


def summary_of(path, ids=None):
    """A probe's measures from its episodes (on the questions in ``ids`` when given)."""
    ep_path = Path(str(path).replace(".json", ".episodes.jsonl"))
    if not ep_path.exists():
        return None
    rows = [r for r in rows_of(ep_path) if ids is None or r["id"] in ids]
    if not rows:
        return None
    sc = [r["score"] for r in rows]
    n = len(rows)
    return {"n": n, "reward": sum(x["total"] for x in sc) / n, "decision_correct": sum(x["decision_correct"] for x in sc) / n,
            "evidence": sum(x["evidence"] for x in sc) / n,
            "answer_found": sum(pc.extract_answer(r["final_text"])[0] is not None for r in rows) / n,
            "recovery": pc.recovery_block([pc.recovery_stats(r["calls"]) for r in rows], sc)}


def paired(d1, d2, tag):
    """d1's reward minus d2's, question by question: the mean and its standard error."""
    p1, p2 = d1 / f"{tag}.episodes.jsonl", d2 / f"{tag}.episodes.jsonl"
    if not (p1.exists() and p2.exists()):
        return None
    a = {r["id"]: r["score"]["total"] for r in rows_of(p1)}
    b = {r["id"]: r["score"]["total"] for r in rows_of(p2)}
    d = [a[i] - b[i] for i in sorted(set(a) & set(b))]
    mu = sum(d) / len(d)
    return mu, (sum((x - mu) ** 2 for x in d) / len(d)) ** 0.5 / len(d) ** 0.5


# MLX's probes: 07's seed and PPO checkpoint under mlx-probes/, an 08 run's in <run>/probes-step<N>/ (one per
# checkpoint, best and last, or one when they are the same step).
SEED_ROW, P07_ROW = f"07: seed {SEED07}", "07: PPO step120"
rows_ = [(SEED_ROW, MLXP / SEED07), (P07_ROW, MLXP / f"07-{PPO07}-step120")]
for tag, run in (("08", PPO08), ("08 D37", PPO08_D37)):
    if run and (RUNS / run).exists():
        for d in sorted((RUNS / run).glob("probes-step*"), key=lambda p: int(p.name.split("step")[1])):
            rows_.append((f"{tag}: PPO {d.name.replace('probes-', '')}", d))
PROBES = (("test", TEST120, "test-T0-n120-stop", "test"), ("holdout_shift", None, "holdout_shift-T0-stop", "holdout_shift"),
          ("dev, forced (greedy)", None, "dev-T0-stop-forced", "forced"), ("dev, 120 at T=1", None, "dev-T1-n120-stop", "T=1"))
for split, ids, tag, _ in PROBES:
    print(f"{split}:")
    for label, d in rows_:
        o = summary_of(d / f"{tag}.json", ids)
        print(f"  {label:26} " + (recov_line(o) if o else "(not run yet)"))
base = dict(rows_)
fmt_p = lambda p: "      -       " if p is None else f"{p[0]:+.3f} ({p[1]:.3f})"  # noqa: E731
print("\nreward, paired by question: the difference (its standard error)")
print(f"  {'':26} {'':8} " + " ".join(f"{short:>15}" for *_, short in PROBES))
for label, d in rows_[1:]:
    for ref in (SEED_ROW, P07_ROW):
        if ref != label:
            print(f"  {label:26} - {'seed' if ref == SEED_ROW else '07 PPO':6} " + " ".join(f"{fmt_p(paired(d, base[ref], tag)):>15}" for _, _, tag, _ in PROBES))

test:
  07: seed seed-n200-r100    n 120 reward 0.564 decision 0.525 evidence 0.475 answers 0.700 | erred 0.483 recovered 0.431 repeats/ep 0.31
  07: PPO step120            n 120 reward 0.630 decision 0.592 evidence 0.575 answers 0.708 | erred 0.508 recovered 0.541 repeats/ep 0.42
  08: PPO step40             n 120 reward 0.654 decision 0.608 evidence 0.617 answers 0.792 | erred 0.333 recovered 0.475 repeats/ep 0.24
  08: PPO step120            n 120 reward 0.612 decision 0.575 evidence 0.558 answers 0.700 | erred 0.475 recovered 0.404 repeats/ep 0.39
holdout_shift:
  07: seed seed-n200-r100    n 120 reward 0.516 decision 0.392 evidence 0.450 answers 0.717 | erred 0.533 recovered 0.531 repeats/ep 0.34
  07: PPO step120            n 120 reward 0.588 decision 0.483 evidence 0.575 answers 0.742 | erred 0.558 recovered 0.567 repeats/ep 0.18
  08: PPO step40             n 120 reward 0.558 decision 0.458 evidence 0.517 answers 0.783 | erred 0.367 recovered 0.500 repeats/ep 0.39
  08: PPO ste

  07: PPO step120            - seed    +0.066 (0.030)  +0.072 (0.024)  +0.072 (0.018)  +0.041 (0.032)


  08: PPO step40             - seed    +0.089 (0.027)  +0.042 (0.019)  +0.007 (0.011)  -0.012 (0.036)


  08: PPO step40             - 07 PPO  +0.024 (0.029)  -0.030 (0.024)  -0.065 (0.016)  -0.052 (0.032)
  08: PPO step120            - seed    +0.048 (0.030)  +0.097 (0.026)  +0.011 (0.018)  +0.012 (0.028)
  08: PPO step120            - 07 PPO  -0.018 (0.024)  +0.025 (0.021)  -0.061 (0.017)  -0.028 (0.031)


## Three seeds: effa2e4's recipe, and with 07's line

The other notebook 07 (Rui Tong's, commit effa2e4) trained its seed differently: the first 300 of 06's reference episodes, every fourth replaced by its own question's recovery episode (225 + 75: 36 unconverted inputs, 39 ×10 / ×0.1 factor slips), one epoch at lr 1e-4, LoRA r16; its recovery episodes fix the call without a line before it. On the same 120 dev questions at T=1 his seed beat 07's (0.490 against 0.430), so 08 trained it here (`seed-rt-n300-e4`, `08_prover_critic_seed.sh`), and the same with 07's diagnosis line opening each fix (`seed-rtfix-n300-e4`, `FIX=1`). Their probes are PyTorch's, as 07's seed's were: the 120 dev questions at T=1 and the forced probe, whose slips are all unconverted inputs.

In [7]:
import re
SEEDS = ((f"07's seed {SEED07} (07's line)", RUNS07 / SEED07 / "epoch1"),
         ("seed-rt-n300-e4 (effa2e4's recipe)", RUNS / "seed-rt-n300-e4" / "epoch1"),
         ("seed-rtfix-n300-e4 (the recipe + 07's line)", RUNS / "seed-rtfix-n300-e4" / "epoch1"))
prefilled = pc.forced_prefill("dev")
for label, d in SEEDS:
    t1, fo = summary_of(d / "dev-T1-n120-stop.json"), summary_of(d / "dev-T0-stop-forced.json")
    if not (t1 and fo):
        print(f"{label}: not run yet")
        continue
    lines = factor = 0
    for r in rows_of(d / "dev-T0-stop-forced.episodes.jsonl"):
        own = r["turns"][len(prefilled[r["id"]]):]  # the turns after the slipped ones: the seed's own
        if own and not own[0].startswith("<tool_call>"):
            lines += 1
            factor += re.search(r"ten times|a tenth of", own[0].split("\n")[0]) is not None
    print(f"{label}:\n   dev, T=1  {recov_line(t1)}\n   forced    {recov_line(fo)}\n"
          f"   forced: its first turn opens with a line in {lines} of {fo['n']}, naming a x10 / x0.1 factor slip in {factor}")

07's seed seed-n200-r100 (07's line):
   dev, T=1  n 120 reward 0.430 decision 0.342 evidence 0.367 answers 0.575 | erred 0.600 recovered 0.306 repeats/ep 0.59
   forced    n 144 reward 0.686 decision 0.632 evidence 0.750 answers 0.764 | erred 1.000 recovered 0.812 repeats/ep 0.67
   forced: its first turn opens with a line in 144 of 144, naming a x10 / x0.1 factor slip in 0
seed-rt-n300-e4 (effa2e4's recipe):
   dev, T=1  n 120 reward 0.489 decision 0.408 evidence 0.425 answers 0.583 | erred 0.425 recovered 0.196 repeats/ep 2.38
   forced    n 144 reward 0.551 decision 0.444 evidence 0.535 answers 0.493 | erred 1.000 recovered 0.562 repeats/ep 4.06
   forced: its first turn opens with a line in 0 of 144, naming a x10 / x0.1 factor slip in 0
seed-rtfix-n300-e4 (the recipe + 07's line):
   dev, T=1  n 120 reward 0.457 decision 0.417 evidence 0.375 answers 0.600 | erred 0.367 recovered 0.159 repeats/ep 0.95
   forced    n 144 reward 0.404 decision 0.250 evidence 0.299 answers 0.299 | err

## Three of the 37 PPO implementation details (D37)

> Huang, Dossa, Raffin, Kanervisto, Wang. *The 37 Implementation Details of Proximal Policy Optimization.* ICLR Blog Track, 2022.

Checked against 03-08's update, two of its core details were missing -- **minibatch updates** (#6: 03-07 take one optimizer step per epoch on the whole batch, 4 a PPO step) and **advantage normalization** (#7) -- and the approximate KL between the rollout policy and the updated one was not logged (#12). 08 adds them as settings, defaults off (03-07's update exactly), and a second run from the same seed takes the details' defaults where they carry over (`D37=1`): advantages normalized per minibatch, 4 minibatches × 4 epochs (16 optimizer steps a PPO step), the gradient-norm clip 0.5, Adam eps 1e-5, the learning rate annealed linearly to 0, no KL early stop. Not taken: the details' learning rate (2.5e-4 / 3e-4) and entropy bonus -- they train small networks from scratch, and here a LoRA of a pretrained 0.8B model ran away at lr 1e-5 (06) -- and γ 0.99 (an episode here ends, its reward at the end). The starting lr stays v1's 2e-6: with four times the optimizer steps, the update per PPO step starts at about four times v1's. Everything else is v1's -- critic, prover, seed, the question at every step, the warm start from 07's first 25 steps -- with MLX generating from the start, so the runs pair by step.

In [8]:
# The update's own measures: v1's steps from 80 on (the code that logs them) against D37's.
for label, run in (("v1 (03-07's update)", R08), ("D37", R37)):
    rows = [r for r in (run["m"] if run else []) if not r["critic_warmup"] and r.get("approx_kl_last") is not None]
    if not rows:
        print(f"{label}: no steps with the update's measures yet")
        continue
    mean = lambda k: sum(r[k] for r in rows if r.get(k) is not None) / max(1, sum(r.get(k) is not None for r in rows))  # noqa: E731
    print(f"{label}, steps {rows[0]['step']}-{rows[-1]['step']}: {mean('updates'):.1f} optimizer steps a PPO step | approx KL to the "
          f"rollout policy: last {mean('approx_kl_last'):.4f}, max {mean('approx_kl_max'):.4f} | first minibatch's ratio "
          f"{mean('ratio_first'):.4f} | clip fraction {mean('clip_frac'):.3f} | entropy {mean('entropy_sampled'):.3f} | grad norm "
          f"{mean('grad_norm'):.3f} | lr {rows[0]['lr']:.2e} -> {rows[-1]['lr']:.2e}")

v1 (03-07's update), steps 80-119: 4.0 optimizer steps a PPO step | approx KL to the rollout policy: last 0.0002, max 0.0002 | first minibatch's ratio 1.0000 | clip fraction 0.001 | entropy 0.020 | grad norm 0.136 | lr 2.00e-06 -> 2.00e-06
D37: no steps with the update's measures yet


## Summary

*(v1 and its held-out probes are in; D37 is running, its readings to come.)*

1. **The critic.** Another model reads 07's episodes about as well as the policy's own features, not better: online q log-loss 0.448 for Qwen3-4B (layer 27) against 0.441 for 07's critic's features, AUC 0.841 against 0.844, v's explained variance 0.392 against 0.390. In PPO the separate critic explained more of the returns than 07's critic did in its run (+0.32 per token against +0.25, the position clock +0.08 in both).
2. **PPO.** From the same seed, on the same questions, v1 did not beat 07: greedy dev 0.679 / 0.592 / 0.622 at steps 40 / 80 / 120 against 0.658 / 0.648 / 0.673, and the sampled reward over the 95 actor steps, paired by step, −0.010 (95% −0.035 to +0.015). Its recovery measures did not move above 07's: erred, then a clean simulate call, 0.40 over its actor steps against 0.45.
3. **Held out** (all MLX, paired by question). Both PPO runs gained on the seed on test and holdout_shift (07: +0.066, +0.072; v1 at step 120: +0.048, +0.097), and v1 against 07 is within noise there (−0.018 ± 0.024, +0.025 ± 0.021). On the forced probe 07's PPO gained +0.072 ± 0.018 and v1's nothing (+0.011 ± 0.018; against 07, −0.061 ± 0.017), at T=1 likewise (+0.041 against +0.012): the prover's progress did not teach the recovery from a simulator error that 07's own advantages did. v1's best evaluation, step 40, is no better: +0.089 on test, +0.007 on the forced probe, −0.012 at T=1.
4. **Seeds.** effa2e4's recipe samples better at T=1 than 07's seed (0.489 against 0.430) but recovers from the forced slips worse (0.562 against 0.812, 4.06 repeated calls an episode against 0.67); with 07's line added it names an unconverted input a ×10 / ×0.1 factor slip in 84 of 144 and recovers least (0.361). 08 kept 07's seed.
5. **MLX.** Generation about 3× faster (median 166 → 53 s a step), the rest of the step unchanged (382 → 390 s): a PPO step 554 → 452 s (median; against v1's quietest PyTorch stretch, steps 61-79 at 486 s, 8% faster). An evaluation of 60 questions took about 7 minutes against about 30 (the run's log), the probes 3-4× less. Greedy, the generator moves the seed's probes by +0.021 ± 0.015 on test (86 of 120 episodes token for token the same) and +0.002 ± 0.006 on the forced probe (118 of 144); at T=1, two independent draws, by +0.066 ± 0.030.
6. **D37.** Running: v1's configuration with three of the 37 details (minibatch advantage normalization, 4 minibatches, the details' clip, eps and annealing).